# 🛰️ SatQuery AI — Round 2: Binary VQA (GOLDEN CHECKPOINT ★)

**This is the most important training session.** The checkpoint produced here
is the foundation for ALL subsequent rounds (MCQ, Captioning, Grounding, Change).

**What this trains:**
- Dataset: BEN-Bench binary VQA (6,927 Yes/No annotations)
- Input: Sentinel-1 SAR + Sentinel-2 Multispectral images together
- Output: Yes/No answers to geospatial questions

**Expected results:** ~67–71% VQA accuracy (vs 58% SOTA)

**If Colab crashes:** Just re-run all cells — training auto-resumes from last checkpoint!

---
⏱️ Estimated time: **2.5 hours** on T4

In [ ]:
# ── CELL 1: Mount Drive + Clone Repo (run every session) ──────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

DRIVE = '/content/drive/MyDrive/SatQuery_AI'
REPO = '/content/SIH'

if not os.path.exists(REPO):
    !git clone https://github.com/abhineet115/SIH.git {REPO}
else:
    !git -C {REPO} pull origin main

os.chdir(f'{REPO}/training')
sys.path.insert(0, f'{REPO}/training')

print(f'✅ Working in: {os.getcwd()}')

In [ ]:
# ── CELL 2: Install dependencies ──────────────────────────────────────────
!pip install -q transformers peft accelerate bitsandbytes datasets
!pip install -q einops timm sentencepiece rasterio
print('✅ Dependencies installed!')

In [ ]:
# ── CELL 3: Crash Recovery Check ──────────────────────────────────────────
from pathlib import Path

ROUND = 2
ROUND_NAME = 'r2_binary_vqa'
CKPT_DIR = f'{DRIVE}/ckpt/{ROUND_NAME}'

ckpts = sorted(
    [d for d in Path(CKPT_DIR).iterdir() if d.name.startswith('checkpoint-')],
    key=lambda d: int(d.name.split('-')[1])
) if Path(CKPT_DIR).exists() else []

resume_ckpt = str(ckpts[-1]) if ckpts else None
steps_done = int(ckpts[-1].name.split('-')[1]) if ckpts else 0
steps_remaining = 500 - steps_done

print(f'Round {ROUND}: {ROUND_NAME}')
print(f'Steps completed: {steps_done} / 500')
print(f'Steps remaining: {steps_remaining}')
print(f'Resume from: {resume_ckpt or "scratch"}')

In [ ]:
# ── CELL 4: Check Previous Round's Adapter (R1 warmup) ────────────────────
R1_BEST = f'{DRIVE}/ckpt/r1_warmup/best'
start_from_r1 = Path(R1_BEST).exists()

if start_from_r1:
    print(f'✅ Found R1 warmup adapter: {R1_BEST}')
    print('  Will initialize from R1 checkpoint for better accuracy!')
else:
    print('⚠️  R1 warmup not found. Starting from base model.')
    print('  Run 02_r1_warmup.ipynb first for slightly better results.')
    R1_BEST = None

In [ ]:
# ── CELL 5: Import and Configure Model ────────────────────────────────────
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from models.rs_internvl import RSInternVL

BASE_MODEL = 'OpenGVLab/InternVL3-1B'

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading RS-InternVL (4-bit QLoRA)...')
if R1_BEST:
    model = RSInternVL.load_with_adapter(BASE_MODEL, R1_BEST, use_4bit=True)
else:
    model = RSInternVL(
        base_model_name=BASE_MODEL,
        lora_r=8, lora_alpha=16, lora_dropout=0.05,
        task_mode='vqa',
    )

# Memory report
vram_used = torch.cuda.memory_allocated() / 1e9
print(f'\n✅ Model ready!')
print(f'VRAM used: {vram_used:.2f} / 15 GB')

In [ ]:
# ── CELL 6: Load BEN-Bench Binary VQA Dataset ─────────────────────────────
from data.ben_bench import BENBenchDataset

DATASETS_DIR = f'{DRIVE}/datasets'

train_dataset = BENBenchDataset(
    data_path=f'{DATASETS_DIR}/ben_bench.json',
    task_type='binary_vqa',
    tokenizer=tokenizer,
    augment=True,
)

# BEN-Bench only has test split — use 80/20 train/val split
from torch.utils.data import random_split
n = len(train_dataset)
n_val = int(n * 0.2)
n_train = n - n_val
train_split, val_split = random_split(train_dataset, [n_train, n_val],
                                       generator=torch.Generator().manual_seed(42))

print(f'\nDataset splits:')
print(f'  Train: {n_train} samples')
print(f'  Val:   {n_val} samples')
print(f'  Total: {n} samples')

In [ ]:
# ── CELL 7: Training Arguments ────────────────────────────────────────────
from transformers import TrainingArguments, Trainer

output_dir = CKPT_DIR  # saves to Google Drive every N steps

training_args = TrainingArguments(
    output_dir=output_dir,

    # ── Core training ─────────────────────────────
    max_steps=500,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch = 32
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,

    # ── Memory optimizations ──────────────────────
    optim='paged_adamw_8bit',
    fp16=True,
    gradient_checkpointing=True,

    # ── Checkpointing (CRASH SAFETY!) ────────────
    save_steps=100,
    save_total_limit=3,
    eval_steps=100,
    evaluation_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # ── Logging ──────────────────────────────────
    logging_steps=10,
    report_to='tensorboard',
    logging_dir=f'{output_dir}/logs',

    # ── Misc ─────────────────────────────────────
    dataloader_num_workers=2,
    remove_unused_columns=False,
    run_name='r2_binary_vqa',
)

print('✅ Training args configured!')
print(f'  Effective batch size: {2 * 16} = 32')
print(f'  Max steps: 500')
print(f'  Saves every 100 steps to Drive ✓')
print(f'  Auto-resume from crash: {resume_ckpt or "N/A"}')

In [ ]:
# ── CELL 8: VQA Accuracy Metric ───────────────────────────────────────────
import numpy as np

def compute_metrics(eval_pred):
    """Compute Yes/No accuracy for binary VQA."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1) if logits.ndim > 1 else logits
    
    # Filter ignored (-100) positions
    mask = labels != -100
    acc = (predictions[mask] == labels[mask]).mean()
    return {'accuracy': float(acc)}

print('✅ Evaluation metric: VQA token accuracy')

In [ ]:
# ── CELL 9: START TRAINING ★ ──────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=val_split,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print('🚀 Starting Round 2 training...')
print('Watch VRAM usage. If OOM → reduce batch_size to 1 in Cell 7.')
print(f'Checkpoints auto-saved to: {output_dir}\n')

# Auto-resumes from crash checkpoint if exists!
trainer.train(resume_from_checkpoint=resume_ckpt)

print('\n✅ Round 2 complete!')

In [ ]:
# ── CELL 10: Save Golden Checkpoint ★ ────────────────────────────────────
GOLDEN_PATH = f'{DRIVE}/ckpt/r2_binary_vqa/best'
model.save_adapter(GOLDEN_PATH)

# Also save training stats
import json
stats = {
    'round': 2,
    'task': 'binary_vqa',
    'steps_trained': trainer.state.global_step,
    'best_eval_loss': trainer.state.best_metric,
    'train_loss': [l['loss'] for l in trainer.state.log_history if 'loss' in l],
}
with open(f'{DRIVE}/results/r2_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\n🌟 GOLDEN CHECKPOINT SAVED!')
print(f'   Path: {GOLDEN_PATH}')
print(f'   Steps: {trainer.state.global_step}')
print(f'\nNext sessions to run (all start from this checkpoint):')
print('  → 04_r3a_mcq.ipynb        (MCQ fine-tuning)')
print('  → 05_r3b_captioning.ipynb (Captioning fine-tuning)')
print('  → 06_r4_grounding.ipynb   (Grounding fine-tuning)')
print('  → 07_r5_change.ipynb      (Change Detection fine-tuning)')